# Robustness Check

The primary econometric analysis found little evidence of a strong or statistically significant relationship between changes in international student enrolments and changes in rental prices in New South Wales and Victoria after controlling for population growth, housing supply, COVID effects, and state fixed effects. The coefficient on international student enrolments was small and sensitive across specifications. This analysis is interpreted as a descriptive analysis of conditional correlations rather than a causal claim.

## Robustness Check 

To support descriptive claims, I include robustness checks that test whether the conditional correlation between international student enrolments and rental prices is driven by a particular subsample, time window, or functional-form choice. Specifically, I estimate the model separately for NSW and VIC, remove the COVID-affected period, and compare the main differenced-log specification with a non-differenced specification.

### Check 1: HC3 Robust Standard Errors

#### Concern

The main regression may have heteroskedasticity, meaning the variance of the error terms may not be constant across observations. If this happens, the usual OLS standard errors may be unreliable, which can affect the p-values and statistical significance of the coefficients. This is especially important in this project because the sample size is small and some quarters may have larger shocks than others.

#### Approach

To address this concern, the regression is re-estimated using HC3 robust standard errors. HC3 adjusts the standard errors to be more reliable when heteroskedasticity or influential observations may be present. The model specification remains the same as the main regression, but the inference is changed by using HC3 standard errors instead of conventional OLS standard errors.

#### Code 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort properly
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create first-differenced log variables
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Create COVID dummy
df["COVID"] = ((df["Date"] >= "2020-Q1") & (df["Date"] <= "2022-Q1")).astype(int)

# Keep complete observations
reg_df = df.dropna(
    subset=[
        "Δ_ln_rent",
        "Δ_ln_students",
        "Δ_ln_population",
        "Δ_ln_supply",
        "COVID",
        "State"
    ]
)

# Robustness Check: HC3 robust standard errors
model_hc3 = smf.ols(
    formula="""
    Q("Δ_ln_rent") ~ Q("Δ_ln_students")
    + Q("Δ_ln_population")
    + Q("Δ_ln_supply")
    + COVID
    + C(State)
    """,
    data=reg_df
).fit(cov_type="HC3")

print(model_hc3.summary())

                            OLS Regression Results                            
Dep. Variable:         Q("Δ_ln_rent")   R-squared:                       0.565
Model:                            OLS   Adj. R-squared:                  0.511
Method:                 Least Squares   F-statistic:                     12.89
Date:                Mon, 11 May 2026   Prob (F-statistic):           1.71e-07
Time:                        21:13:30   Log-Likelihood:                 148.15
No. Observations:                  46   AIC:                            -284.3
Df Residuals:                      40   BIC:                            -273.3
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.0295 

### Check 2: Remove the COVID Dummy

#### Concern


The main regression may rely heavily on the COVID dummy specification. Since the pandemic period caused major disruptions to rental markets, migration flows, border policies, and economic activity, the estimated relationship between international student enrolments and rental prices may be sensitive to how the COVID period is controlled for.

#### Approach 

To test this, the regression is re-estimated after removing the COVID dummy variable while keeping the remaining specification unchanged. If the coefficient on changes in international student enrolments remains similar in sign, magnitude, and statistical significance, this suggests that the main result is not driven solely by the inclusion of the COVID control.

#### Code

In [19]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort properly
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create first-differenced log variables
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Keep complete observations
reg_df = df.dropna(
    subset=[
        "Δ_ln_rent",
        "Δ_ln_students",
        "Δ_ln_population",
        "Δ_ln_supply",
        "State"
    ]
)

# Robustness Check: Remove COVID dummy
model_no_covid = smf.ols(
    formula="""
    Q("Δ_ln_rent") ~ Q("Δ_ln_students")
    + Q("Δ_ln_population")
    + Q("Δ_ln_supply")
    + C(State)
    """,
    data=reg_df
).fit()

print(model_no_covid.summary())

                            OLS Regression Results                            
Dep. Variable:         Q("Δ_ln_rent")   R-squared:                       0.504
Model:                            OLS   Adj. R-squared:                  0.456
Method:                 Least Squares   F-statistic:                     10.42
Date:                Mon, 11 May 2026   Prob (F-statistic):           6.43e-06
Time:                        21:14:31   Log-Likelihood:                 145.12
No. Observations:                  46   AIC:                            -280.2
Df Residuals:                      41   BIC:                            -271.1
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.0195 

### Check 3: Subsample Analysis by State

#### Concern

The main regression pools NSW and VIC into one model. However, the relationship between international student enrolments and rental prices may differ across states because NSW and VIC have different rental markets, population patterns, and student concentrations. Therefore, the pooled result may hide state-specific differences or be driven mainly by one state.

#### Approach

To test this, I re-estimate the main regression separately for NSW and VIC. This keeps the same variables as the main specification, but removes state fixed effects because each regression only contains one state. The purpose is to check whether the coefficient on Δ_ln_students remains similar across both states.

#### Code

In [20]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort data
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create first-differenced log variables
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Create COVID dummy
df["COVID"] = ((df["Date"] >= "2020-Q1") & (df["Date"] <= "2022-Q1")).astype(int)

# Keep complete observations
reg_df = df.dropna(
    subset=[
        "Δ_ln_rent",
        "Δ_ln_students",
        "Δ_ln_population",
        "Δ_ln_supply",
        "COVID"
    ]
)

# Run separate regressions for NSW and VIC
model_nsw = smf.ols(
    "Q('Δ_ln_rent') ~ Q('Δ_ln_students') + Q('Δ_ln_population') + Q('Δ_ln_supply') + COVID",
    data=reg_df[reg_df["State"] == "NSW"]
).fit()

model_vic = smf.ols(
    "Q('Δ_ln_rent') ~ Q('Δ_ln_students') + Q('Δ_ln_population') + Q('Δ_ln_supply') + COVID",
    data=reg_df[reg_df["State"] == "VIC"]
).fit()

# Print results
print("NSW Only Regression")
print(model_nsw.summary())

print("\n" + "="*80 + "\n")

print("VIC Only Regression")
print(model_vic.summary())

NSW Only Regression
                            OLS Regression Results                            
Dep. Variable:         Q('Δ_ln_rent')   R-squared:                       0.558
Model:                            OLS   Adj. R-squared:                  0.459
Method:                 Least Squares   F-statistic:                     5.675
Date:                Mon, 11 May 2026   Prob (F-statistic):            0.00390
Time:                        21:25:43   Log-Likelihood:                 73.341
No. Observations:                  23   AIC:                            -136.7
Df Residuals:                      18   BIC:                            -131.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept   

### Check 4: Logged Level Specification

#### Concern

The first-differenced log specification may remove important long-run variation and persistent housing market relationships between international student enrolments and rental prices. As a result, the main model may understate broader long-run associations between the variables.

#### Approach

Estimate the regression using logged level variables rather than first-differenced logged variables to assess whether the main findings are sensitive to the removal of long-run variation through differencing.

#### Code

In [21]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create COVID dummy
df["COVID"] = (
    (df["Date"] >= "2020-Q1") &
    (df["Date"] <= "2022-Q1")
).astype(int)

# Run logged-level regression
model_logged_levels = smf.ols(
    "ln_rent ~ ln_students + ln_population + ln_supply + C(State) + COVID",
    data=df
).fit()

# Print results
print(model_logged_levels.summary())

                            OLS Regression Results                            
Dep. Variable:                ln_rent   R-squared:                       0.975
Model:                            OLS   Adj. R-squared:                  0.971
Method:                 Least Squares   F-statistic:                     321.3
Date:                Mon, 11 May 2026   Prob (F-statistic):           2.58e-32
Time:                        22:24:27   Log-Likelihood:                 111.85
No. Observations:                  48   AIC:                            -211.7
Df Residuals:                      42   BIC:                            -200.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         -67.3853      4.971    -

## Robustness Table 

| Variables                         | ( 1) Main Specification | (2) HC3 Robust SE | (3) Remove COVID Dummy | (4) NSW Only | (5) VIC Only | (6) Without Differencing |
| --------------------------------- | ----------------------- | ----------------- | ---------------------- | ------------ | ------------ | ------------------------ |
| Δ ln(Students) / ln(Students)     | 0.0003                  | 0.0003            | -0.0009                | -0.0003      | 0.0009       | -0.0020                  |
|                                   | (0.001)                 | (0.001)           | (0.001)                | (0.002)      | (0.002)      | (0.004)                  |
| Δ ln(Population) / ln(Population) | 1.3973                  | 1.3973            | 2.9870                 | 1.4137       | 1.6066       | 4.9648                   |
|                                   | (1.009)                 | (1.009)           | (0.707)                | (2.323)      | (1.260)      | (0.435)                  |
| Δ ln(Supply) / ln(Supply)         | -4.1544                 | -4.1544           | -3.8533                | -4.3678      | -2.8857      | -0.6652                  |
|                                   | (2.595)                 | (2.595)           | (2.475)                | (4.338)      | (3.863)      | (0.286)                  |
| COVID                             | -0.0115                 | -0.0115           | —                      | -0.0120      | -0.0099      | -0.0067                  |
|                                   | (0.005)                 | (0.005)           | —                      | (0.007)      | (0.008)      | (0.010)                  |
| State Fixed Effects               | Yes                     | Yes               | Yes                    | No           | No           | Yes                      |
| First-Differenced Logs            | Yes                     | Yes               | Yes                    | Yes          | Yes          | No                       |
| Robust Standard Errors            | No                      | Yes (HC3)         | No                     | No           | No           | No                       |
| Sample Restriction                | Full Sample             | Full Sample       | Full Sample            | NSW Only     | VIC Only     | Full Sample              |
| Observations (N)                  | 46                      | 46                | 46                     | 23           | 23           | 48                       |
| R²                                | 0.565                   | 0.565             | 0.504                  | 0.558        | 0.583        | 0.975                    |

**Notes**: Outcome is Δln(Rent) except col. (6), where the outcome is ln(Rent) in levels. All columns control for population and housing supply. Cols. (1), (2), (4), (5), and (6) include a COVID dummy, while col. (3) removes the COVID control. Cols. (1)–(3) and (6) include state fixed effects. Col. (2) re-estimates the baseline model using HC3 heteroskedasticity-robust standard errors. Cols. (4) and (5) estimate the model separately for NSW and VIC. Col. (6) re-estimates the specification without first differencing. Standard errors in parentheses.


## Interpretation

Overall, the robustness checks suggest that the main finding is reasonably stable: changes in international student enrolments do not show a strong or statistically meaningful relationship with changes in rental prices once broader demographic and housing factors are controlled for. Across most specifications, the coefficient on student enrolments remains extremely small and changes sign depending on the specification, indicating that the relationship is not robustly positive or negative.

The HC3 robust standard error specification in column (2) produces virtually identical estimates to the main specification in column (1). This suggests that heteroskedasticity is not materially affecting the inference, strengthening the reliability of the baseline standard errors and supporting the credibility of the original result.

Removing the COVID dummy in column (3) slightly changes the coefficient on student enrolments from positive to negative, although the estimate remains very close to zero. The reduction in R^2 from 0.565 to 0.504 also suggests that the COVID period captures an important common shock affecting rental markets. However, the absence of a strong student effect even after removing the COVID control indicates that the baseline finding is not solely driven by the pandemic adjustment.

The NSW-only and VIC-only regressions in columns (4) and (5) also produce very small coefficients with different signs across states. This suggests that the estimated relationship is not being driven entirely by one state. Although the coefficients vary somewhat across subsamples, the lack of a consistently large or statistically meaningful effect across both states reinforces the conclusion that short-run changes in student numbers are not strongly associated with rental price changes in this dataset.

The largest change occurs in column (6), where the model is estimated in log levels rather than first-differenced logs. The coefficient on student enrolments becomes slightly larger in magnitude and negative, while the R^2 rises substantially to 0.975. This likely reflects the presence of strong common upward trends in both rents and macroeconomic variables over time. Because the levels specification does not remove non-stationary trends, the very high R^2 may partly capture shared time movements rather than meaningful economic relationships. This supports the decision to use first-differenced logs in the main specification, as differencing provides a more conservative and credible estimate of the short-run association.

The robustness checks indicate that the main conclusion survives across alternative inference methods, sample restrictions, and control choices. While the exact coefficient changes modestly across specifications, there is no consistent evidence of a strong positive relationship between international student enrolments and rental prices after accounting for population growth, housing supply, and broader time effects.